# Classifying Dataset Reuse vs Primary Data Generation in GEO-related Papers

This notebook implements an end-to-end pipeline to classify whether a biomedical paper:

- **Primary**: generated its own dataset (even if later deposited to GEO/SRA), or
- **Reuse**: reused an existing public dataset (e.g. downloaded from GEO).

Pipeline steps:

1. Read papers from a **JSONL** file (one paper per line)
2. Build a **provenance evidence bundle** (windows around accessions + provenance verbs)
3. Call **Ollama** with **streaming** to classify (strict JSON)
4. Save predictions to CSV
5. Evaluate against a manually labeled CSV (ground truth) if provided
6. Produce plots: accuracy, coverage/unclear rate, confusion matrices


## Why evidence bundles?

Titles and abstracts describe scientific findings, not data provenance.
Provenance signals usually appear in:

- **Data availability** statements
- **Methods / Materials** sections
- Sentences near dataset accessions (e.g., **GSE12345**)

Instead of giving the LLM the whole paper, we extract only provenance-relevant snippets and require the model to **quote** the evidence it relied on.


In [1]:
import json, re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt


## Configuration

Edit these paths and settings for your environment.


In [2]:
# -------- Paths --------
JSONL_PATH = Path("pmc_gse_articles_clean.jsonl")  # <-- change
LABELED_CSV_PATH = Path("manual_ground_truth_with_GSE_links_REFRESHED.csv")  # optional; change or set to None

OUT_DIR = Path("outputs_notebook")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -------- Ollama settings --------
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3"  # e.g. "llama3", "llama3:8b", etc.

# LLM call mode:
# - "unclear_only": only call LLM when heuristic says Unclear (recommended)
# - "all": call LLM for every paper
# - "off": no LLM calls
LLM_MODE = "unclear_only"

# Evidence extraction controls
WIN_BEFORE = 350
WIN_AFTER = 900
MAX_EVIDENCE_CHARS = 2200

# Ollama generation controls
TEMPERATURE = 0.0
NUM_PREDICT = 220
TIMEOUT_CONNECT = 10
TIMEOUT_READ = 600


## Patterns: accessions and provenance cues


In [3]:
ACC_PATTERNS = {
    "GEO_GSE": re.compile(r"\bGSE\d+\b", re.IGNORECASE),
    "GEO_GSM": re.compile(r"\bGSM\d+\b", re.IGNORECASE),
    "SRA_SRP": re.compile(r"\bSRP\d+\b", re.IGNORECASE),
    "SRA_SRR": re.compile(r"\bSRR\d+\b", re.IGNORECASE),
    "ENA_PRJ": re.compile(r"\bPRJ[EDNA][A-Z0-9]+\b", re.IGNORECASE),
    "ArrayExpress": re.compile(r"\bE-\w{2,3}-\d+\b", re.IGNORECASE),
}
ACC_ANY = re.compile(r"(?i)\b(GSE\d+|GSM\d+|SRP\d+|SRR\d+|E-\w{2,3}-\d+|PRJ[EDNA][A-Z0-9]+)\b")

GEO_WORDS = re.compile(r"(?i)\b(GEO|Gene Expression Omnibus|SRA|ArrayExpress|ENA|NCBI)\b")
PROV_WORDS = re.compile(
    r"(?i)\b(downloaded|retrieved|obtained|reanaly[sz]ed|re-analysed|publicly available|"
    r"deposited|submitted|accession|available at|data availability|data are available)\b"
)

# Triage cues (conservative)
PRIMARY_STRONG = re.compile(
    r"(?i)\b(we (collected|recruited|enrolled|sequenced|generated|performed rna-?seq|acquired)|"
    r"library preparation|sample collection|patients were recruited|ethics approval|informed consent|our cohort)\b"
)
REUSE_STRONG = re.compile(
    r"(?i)\b(downloaded|retrieved|obtained from|publicly available|reanaly[sz]ed|secondary analysis)\b"
)
DEPOSIT = re.compile(r"(?i)\b(deposited|submitted)\b")
WE_OUR = re.compile(r"(?i)\b(we|our)\b")


# Data availability heading cues (optional but useful)
DATA_AVAIL = re.compile(r"(?i)\b(data availability|availability of data|data and materials availability|availability of data and materials)\b")


## Helper functions: text extraction, evidence bundle, heuristic triage


In [7]:
def extract_text_fields(rec: Dict[str, Any]) -> str:
    """"Adjust keys if your JSON schema differs."""
    parts: List[str] = []
    for k in ["title", "abstract", "full_text", "body", "text"]:
        v = rec.get(k)
        if isinstance(v, str) and v.strip():
            parts.append(v.strip())
    return "\\n\\n".join(parts)

def extract_accessions(text: str) -> Dict[str, List[str]]:
    t = text or ""
    acc: Dict[str, List[str]] = {}
    for k, pat in ACC_PATTERNS.items():
        acc[k] = sorted(set(m.group(0).upper() for m in pat.finditer(t)))
    return acc

def format_accession_string(acc: Dict[str, List[str]], per_key_cap: int = 10) -> str:
    parts = []
    for k, v in acc.items():
        if v:
            parts.append(f"{k}: {', '.join(v[:per_key_cap])}")
    return "; ".join(parts) if parts else "None"

def extract_data_availability(text: str, max_chars: int = 1200) -> str:
    """ 
    Extract a short window starting at the first Data Availability heading (if present).
    This is a best-effort heuristic: PMC plain text can be messy, so we keep it simple.
    """
    flat = re.sub(r"\s+", " ", (text or "")).strip()
    if not flat:
        return ""
    m = DATA_AVAIL.search(flat)
    if not m:
        return ""
    return flat[m.start(): m.start() + int(max_chars)]

def evidence_windows(
    text: str,
    win_before: int = 350,
    win_after: int = 900,
    max_chars: int = 2200,
    max_hits: int = 40
    ) -> Tuple[List[str], str]:
    """
    Evidence bundle = windows around accession mentions;
    fallback to provenance-verb windows if no accession windows are found.
    """
    flat = re.sub(r"\s+", " ", (text or "")).strip()
    if not flat:
        return [], ""

    hits: List[str] = []

    # (A) Windows around accession mentions
    for m in ACC_ANY.finditer(flat):
        s = max(0, m.start() - win_before)
        e = min(len(flat), m.end() + win_after)
        chunk = flat[s:e]
        if GEO_WORDS.search(chunk) or PROV_WORDS.search(chunk):
            hits.append(chunk)
        if len(hits) >= max_hits:
            break

    # (B) Fallback: windows around provenance verbs
    if not hits:
        for m in PROV_WORDS.finditer(flat):
            s = max(0, m.start() - win_before)
            e = min(len(flat), m.end() + win_after)
            hits.append(flat[s:e])
            if len(hits) >= max_hits:
                break

    # Deduplicate overlapping chunks
    dedup: List[str] = []
    seen = set()
    for h in hits:
        key = h[:140].lower()
        if key in seen:
            continue
        seen.add(key)
        dedup.append(h)

    # Prioritize chunks that contain provenance verbs
    dedup.sort(key=lambda c: PROV_WORDS.search(c) is not None, reverse=True)

    evidence = " ... ".join(dedup)
    evidence = re.sub(r"\s+", " ", evidence).strip()

    return dedup, evidence[:max_chars]


def heuristic_triage(evidence: str, accessions: Dict[str, List[str]]) -> Tuple[str, float]:
    ev = evidence or ""
    if REUSE_STRONG.search(ev):
        return "Reuse", 0.85
    if PRIMARY_STRONG.search(ev):
        return "Primary", 0.80
    if DEPOSIT.search(ev) and WE_OUR.search(ev) and any(accessions.values()):
        return "Primary", 0.65
    return "Unclear", 0.50


In [13]:
##Extend of 
DATA_NOUNS = re.compile(r"(?i)\b(data|dataset|datasets|samples)\b")
SOURCE_CUES = re.compile(
    r"(?i)\b(from|obtained from|downloaded from|retrieved from|"
    r"publicly available|available at|deposited in|deposited to|submitted to)\b"
)

METHODS_HINT = re.compile(r"(?i)\b(methods|methadology|materials and methods|experimental procedures)\b")


def evidence_windows(
    text: str,
    win_before: int = 300,
    win_after: int = 700,
    max_chars: int = 2200,
    max_hits: int = 40,
) -> Tuple[List[str], str]:

    flat = re.sub(r"\s+", " ", (text or "")).strip()
    if not flat:
        return [], ""

    hits = []

    # (1) Accession-anchored windows (unchanged)
    for m in ACC_ANY.finditer(flat):
        s = max(0, m.start() - win_before)
        e = min(len(flat), m.end() + win_after)
        chunk = flat[s:e]
        if GEO_WORDS.search(chunk) or PROV_WORDS.search(chunk):
            hits.append(chunk)
        if len(hits) >= max_hits:
            break

    # (2) Provenance-verb windows (unchanged)
    if not hits:
        for m in PROV_WORDS.finditer(flat):
            s = max(0, m.start() - win_before)
            e = min(len(flat), m.end() + win_after)
            hits.append(flat[s:e])
            if len(hits) >= max_hits:
                break

    # (3) NEW: data/dataset + source cue, preferably near Methods
    if not hits:
        for m in DATA_NOUNS.finditer(flat):
            s = max(0, m.start() - win_before)
            e = min(len(flat), m.end() + win_after)
            chunk = flat[s:e]

            # Require source cue
            if not SOURCE_CUES.search(chunk):
                continue

            # Optional: prioritize chunks near Methods
            score = 0
            if METHODS_HINT.search(chunk):
                score += 1
            if PROV_WORDS.search(chunk):
                score += 2

            hits.append((score, chunk))
            if len(hits) >= max_hits:
                break

        # keep best-scoring chunks
        hits = [c for _, c in sorted(hits, key=lambda x: x[0], reverse=True)]

    # Deduplicate + prioritize provenance
    dedup = []
    seen = set()
    for h in hits:
        key = h[:140].lower()
        if key in seen:
            continue
        seen.add(key)
        dedup.append(h)

    dedup.sort(key=lambda c: PROV_WORDS.search(c) is not None, reverse=True)

    evidence = " ... ".join(dedup)
    evidence = re.sub(r"\s+", " ", evidence).strip()

    return dedup, evidence[:max_chars]


## LLM prompt + Ollama streaming call


In [8]:
def build_prompt(accessions_str: str, snippets: List[str], evidence_text: str) -> str:
    snippet_block = "\n".join([f"- {s[:350]}" for s in snippets[:8]]) if snippets else "(none)"

    return f"""You are classifying whether a paper GENERATED its dataset (Primary) or REUSED a public/external dataset (Reuse).

Definitions:
- Primary: authors generated the dataset used for analysis (even if they later deposited it to GEO/SRA).
- Reuse: authors downloaded/obtained public/external datasets (e.g., from GEO/SRA/ArrayExpress) and analyzed them.
If evidence is insufficient, return "Unclear".

Accessions mentioned: {accessions_str}

Evidence snippets (near accessions / provenance statements):
{snippet_block}

Condensed evidence text:
\"\"\"{evidence_text}\"\"\"

Output STRICT JSON only (no extra text):
{{
  "label": "Primary" | "Reuse" | "Unclear",
  "confidence": 0.0-1.0,
  "justification": "<=25 words, must quote key phrases from evidence"
}}
"""


def ollama_generate_stream(
    prompt: str,
    model: str,
    url: str,
    temperature: float = 0.0,
    num_predict: int = 220,
    timeout_connect: int = 10,
    timeout_read: int = 600
) -> str:
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": True,
        "options": {
            "temperature": float(temperature),
            "num_predict": int(num_predict),
        },
    }

    r = requests.post(url, json=payload, stream=True, timeout=(timeout_connect, timeout_read))
    r.raise_for_status()

    chunks: List[str] = []
    for line in r.iter_lines(decode_unicode=True):
        if not line:
            continue
        j = json.loads(line)
        if "response" in j:
            chunks.append(j["response"])
        if j.get("done"):
            break

    return "".join(chunks).strip()


def parse_llm_json(s: str) -> Tuple[Optional[str], Optional[float], Optional[str]]:
    if not s:
        return None, None, None

    # Try direct JSON first
    try:
        j = json.loads(s)
        label = j.get("label")
        conf = j.get("confidence")
        just = j.get("justification")
        return label, (float(conf) if conf is not None else None), just
    except Exception:
        pass

    # Fallback: extract first JSON object from text
    m = re.search(r"\{.*\}", s, flags=re.DOTALL)
    if not m:
        return None, None, s[:400]

    try:
        j = json.loads(m.group(0))
        label = j.get("label")
        conf = j.get("confidence")
        just = j.get("justification")
        return label, (float(conf) if conf is not None else None), just
    except Exception:
        return None, None, s[:400]



## Run inference: JSONL → predictions.csv

This cell:
- iterates over all papers
- extracts evidence bundle
- applies heuristic triage
- calls LLM depending on `LLM_MODE`
- writes `predictions.csv` into `OUT_DIR`


In [18]:
def iter_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)

pred_rows = []
n_total = 0
n_llm = 0

for rec in iter_jsonl(JSONL_PATH):
    n_total += 1
    paper_id = rec.get("paper_id") or rec.get("pmcid") or rec.get("doi") or rec.get("id") or ""
    text = extract_text_fields(rec)

    acc = extract_accessions(text)
    acc_str = format_accession_string(acc)

    snippets, evidence = evidence_windows(
        text,
        win_before=WIN_BEFORE,
        win_after=WIN_AFTER,
        max_chars=MAX_EVIDENCE_CHARS,
    )

    # (Optional) prioritize Data Availability statements by prepending them to evidence
    da_text = extract_data_availability(text, max_chars=1200)
    if da_text:
        evidence = (da_text + " ... " + evidence)[:MAX_EVIDENCE_CHARS]


    h_label, h_conf = heuristic_triage(evidence, acc)

    llm_label = llm_conf = llm_just = None
    if LLM_MODE != "off":
        do_llm = (LLM_MODE == "all") or (LLM_MODE == "unclear_only" and h_label == "Unclear")
        if do_llm:
            prompt = build_prompt(acc_str, snippets, evidence)
            try:
                out = ollama_generate_stream(
                    prompt=prompt,
                    model=OLLAMA_MODEL,
                    url=OLLAMA_URL,
                    temperature=TEMPERATURE,
                    num_predict=NUM_PREDICT,
                    timeout_connect=TIMEOUT_CONNECT,
                    timeout_read=TIMEOUT_READ,
                )
                llm_label, llm_conf, llm_just = parse_llm_json(out)
                n_llm += 1
            except Exception as e:
                llm_label, llm_conf, llm_just = None, None, f"LLM_ERROR: {type(e).__name__}: {e}"

    pred_rows.append({
        "paper_id": paper_id,
        "gse_ids": ",".join(acc.get("GEO_GSE", [])),
        "accessions": acc_str,
        "heuristic_label": h_label,
        "heuristic_conf": h_conf,
        "llm_label": llm_label,
        "llm_conf": llm_conf,
        "llm_justification": llm_just,
        "evidence_text": evidence,
    })

    if n_total % 50 == 0:
        print(f"[progress] processed={n_total} llm_calls={n_llm}")

preds = pd.DataFrame(pred_rows)
pred_csv = OUT_DIR / "predictions_extend2.csv"
preds.to_csv(pred_csv, index=False)

print("Wrote:", pred_csv, "rows:", len(preds), "llm_calls:", n_llm)
preds.head()


[progress] processed=50 llm_calls=27
[progress] processed=100 llm_calls=40
[progress] processed=150 llm_calls=58
[progress] processed=200 llm_calls=79
[progress] processed=250 llm_calls=102
Wrote: outputs_notebook\predictions_extend2.csv rows: 250 llm_calls: 102


,paper_id,gse_ids,accessions,heuristic_label,heuristic_conf,llm_label,llm_conf,llm_justification,evidence_text
0,PMC5352056,"GSE81996,GSE81997","GEO_GSE: GSE81996, GSE81997",Unclear,0.50,Primary,1.0,The authors generated the dataset by labeling ...,Technologies). RNA samples were then labeled a...
1,PMC7609712,GSE144348,GEO_GSE: GSE144348,Reuse,0.85,None,NaN,None,analysis was performed using the DESeq2 algori...
2,PMC2244803,GSE8396,GEO_GSE: GSE8396,Unclear,0.50,Primary,1.0,All microarray results have been deposited int...,andard errors have been moderated across genes...
3,PMC2884545,GSE19325,GEO_GSE: GSE19325,Reuse,0.85,None,NaN,None,"We obtained 20,041,035 and 9,780,523 total rea..."
4,PMC4264613,"GSE20711,GSE20712,GSE31979,GSE32393,GSE32838,G...","GEO_GSE: GSE20711, GSE20712, GSE31979, GSE3239...",Reuse,0.85,None,NaN,None,is estimable by our recently published statist...


In [17]:
def iter_jsonl(path: Path):
    # safer reader: tolerate occasional bad lines
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError as e:
                # skip bad line but keep going
                print(f"[warn] JSON decode failed at line {i}: {e}")
                continue


pred_rows = []
n_total = 0
n_llm = 0

for rec in iter_jsonl(JSONL_PATH):
    n_total += 1

    paper_id = rec.get("paper_id") or rec.get("pmcid") or rec.get("doi") or rec.get("id") or ""
    text = extract_text_fields(rec)

    acc = extract_accessions(text)
    acc_str = format_accession_string(acc)

    snippets, evidence = evidence_windows(
        text,
        win_before=WIN_BEFORE,
        win_after=WIN_AFTER,
        max_chars=MAX_EVIDENCE_CHARS,
    )

    # Optional: prepend Data Availability (still capped)
    da_text = extract_data_availability(text, max_chars=1200)
    if da_text:
        evidence = (da_text + " ... " + (evidence or "")).strip()
    evidence = (evidence or "")[:MAX_EVIDENCE_CHARS]

    # heuristic triage (keep it for comparison, but DO NOT gate LLM)
    h_label, h_conf = heuristic_triage(evidence, acc)

    # ---- ALWAYS call LLM ----
    llm_raw = ""
    llm_label = "Unclear"      # conservative default to keep output consistent
    llm_conf = None
    llm_just = ""
    llm_status = "ERROR"

    prompt = build_prompt(acc_str, snippets, evidence)

    try:
        llm_raw = ollama_generate_stream(
            prompt=prompt,
            model=OLLAMA_MODEL,
            url=OLLAMA_URL,
            temperature=TEMPERATURE,
            num_predict=NUM_PREDICT,
            timeout_connect=TIMEOUT_CONNECT,
            timeout_read=TIMEOUT_READ,
        )

        # parse output
        plabel, pconf, pjust = parse_llm_json(llm_raw)

        if plabel is None:
            llm_status = "PARSE_FAIL"
            llm_label = "Unclear"
            llm_conf = None
            llm_just = (pjust or llm_raw[:300] or "Could not parse JSON; defaulted to Unclear.")
        else:
            llm_status = "OK"
            llm_label = plabel
            llm_conf = pconf
            llm_just = pjust or ""

        n_llm += 1

    except Exception as e:
        llm_status = "LLM_ERROR"
        llm_label = "Unclear"
        llm_conf = None
        llm_just = f"LLM_ERROR: {type(e).__name__}: {e}"

    pred_rows.append({
        "paper_id": paper_id,
        "gse_ids": ",".join(acc.get("GEO_GSE", [])),
        "accessions": acc_str,

        "heuristic_label": h_label,
        "heuristic_conf": h_conf,

        "llm_label": llm_label,
        "llm_conf": llm_conf,
        "llm_justification": llm_just,
        "llm_status": llm_status,
        "llm_raw": llm_raw[:2000],  # keep short to avoid huge CSV

        "evidence_text": evidence,
    })

    if n_total % 50 == 0:
        print(f"[progress] processed={n_total} llm_calls={n_llm}")

preds = pd.DataFrame(pred_rows)
pred_csv = OUT_DIR / "predictions_extend.csv"
preds.to_csv(pred_csv, index=False, encoding="utf-8")

print("Wrote:", pred_csv, "rows:", len(preds), "llm_calls:", n_llm)
preds.head()


[progress] processed=50 llm_calls=50
[progress] processed=100 llm_calls=100
[progress] processed=150 llm_calls=150
[progress] processed=200 llm_calls=200
[progress] processed=250 llm_calls=250
Wrote: outputs_notebook\predictions_extend.csv rows: 250 llm_calls: 250


,paper_id,gse_ids,accessions,heuristic_label,heuristic_conf,llm_label,llm_conf,llm_justification,llm_status,llm_raw,evidence_text
0,PMC5352056,"GSE81996,GSE81997","GEO_GSE: GSE81996, GSE81997",Unclear,0.50,Primary,1.0,The authors generated the dataset by labeling ...,OK,"{\n ""label"": ""Primary"",\n ""confidence"": 1.0,...",Technologies). RNA samples were then labeled a...
1,PMC7609712,GSE144348,GEO_GSE: GSE144348,Reuse,0.85,Primary,0.8,Experimental data have been uploaded into the ...,OK,"{\n""label"": ""Primary"",\n""confidence"": 0.8,\n""j...",analysis was performed using the DESeq2 algori...
2,PMC2244803,GSE8396,GEO_GSE: GSE8396,Unclear,0.50,Primary,1.0,All microarray results have been deposited int...,OK,Here is the output in STRICT JSON format:\n\n{...,andard errors have been moderated across genes...
3,PMC2884545,GSE19325,GEO_GSE: GSE19325,Reuse,0.85,Primary,1.0,We obtained... Illumina Genome Analyzer II for...,OK,"{\n ""label"": ""Primary"",\n ""confidence"": 1.0,...","We obtained 20,041,035 and 9,780,523 total rea..."
4,PMC4264613,"GSE20711,GSE20712,GSE31979,GSE32393,GSE32838,G...","GEO_GSE: GSE20711, GSE20712, GSE31979, GSE3239...",Reuse,0.85,Reuse,1.0,The authors obtained three publicly available ...,OK,"{\n ""label"": ""Reuse"",\n ""confidence"": 1.0,\n...",is estimable by our recently published statist...


## Evaluation vs ground truth (optional)

If you have a labeled CSV, this cell merges predictions by `paper_id`,
computes metrics, and saves plots.

It expects `human_label` (or `ground_truth`) as the manual label column.
Legacy columns `llm_label` and `heuristic_label` (v1) will be used if present.


In [ ]:
def normalize_label(x: Any) -> Any:
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in ["primary", "p", "generated", "own", "new"] or "primary" in s:
        return "Primary"
    if s in ["reuse", "re-used", "reused", "secondary", "public", "old"] or "reuse" in s:
        return "Reuse"
    if s in ["unclear", "unknown", "na", "n/a", ""] or "unclear" in s:
        return "Unclear"
    return str(x)

def compute_metrics(y_true: pd.Series, y_pred: pd.Series) -> Dict[str, Any]:
    mask = y_true.isin(["Primary", "Reuse"]) & y_pred.isin(["Primary", "Reuse"])
    yt = y_true[mask]
    yp = y_pred[mask]
    if len(yt) == 0:
        return {"n": 0, "accuracy": np.nan, "precision_primary": np.nan, "recall_primary": np.nan, "f1_primary": np.nan}

    acc = float((yt == yp).mean())
    tp = int(((yp == "Primary") & (yt == "Primary")).sum())
    fp = int(((yp == "Primary") & (yt == "Reuse")).sum())
    fn = int(((yp == "Reuse") & (yt == "Primary")).sum())

    prec = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    rec = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    f1 = (2 * prec * rec / (prec + rec)) if (prec == prec and rec == rec and (prec + rec) > 0) else np.nan

    return {
        "n": int(len(yt)),
        "accuracy": acc,
        "precision_primary": float(prec) if prec == prec else np.nan,
        "recall_primary": float(rec) if rec == rec else np.nan,
        "f1_primary": float(f1) if f1 == f1 else np.nan,
    }

def confusion_matrix(y_true: pd.Series, y_pred: pd.Series) -> Tuple[np.ndarray, int]:
    labels = ["Primary", "Reuse"]
    m = np.zeros((2, 2), dtype=int)
    mask = y_true.isin(labels) & y_pred.isin(labels)
    yt = y_true[mask]
    yp = y_pred[mask]
    for i, t in enumerate(labels):
        for j, p in enumerate(labels):
            m[i, j] = int(((yt == t) & (yp == p)).sum())
    return m, int(mask.sum())

def plot_outputs(metrics_df: pd.DataFrame,
                 df_eval: pd.DataFrame,
                 models: Dict[str, str],
                 out_dir: Path) -> Dict[str, Path]:

    fig1 = out_dir / "accuracy_comparison.png"
    fig2 = out_dir / "coverage_unclear_comparison.png"
    fig3 = out_dir / "confusion_matrices.png"

    order = list(metrics_df.index)
    x = np.arange(len(order))

    # Accuracy
    acc = metrics_df["accuracy"].values
    n = metrics_df["n"].values
    plt.figure(figsize=(10, 4))
    plt.bar(x, np.nan_to_num(acc, nan=0.0))
    plt.xticks(x, order, rotation=20, ha="right")
    plt.ylim(0, 1)
    plt.ylabel("Accuracy (binary only)")
    plt.title("Accuracy vs Ground Truth (Primary/Reuse) on Evaluable Rows")
    for i, (a, ni) in enumerate(zip(acc, n)):
        txt = "n=0" if np.isnan(a) else f"n={ni}\\n{a:.2f}"
        plt.text(i, 0.02, txt, ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plt.savefig(fig1, dpi=200)
    plt.close()

    # Coverage / Unclear
    cov = metrics_df["coverage_binary"].values
    unc = metrics_df["unclear_rate"].values
    w = 0.35
    plt.figure(figsize=(10, 4))
    plt.bar(x - w/2, cov, width=w, label="Coverage (Primary/Reuse predictions)")
    plt.bar(x + w/2, unc, width=w, label="Unclear rate")
    plt.xticks(x, order, rotation=20, ha="right")
    plt.ylim(0, 1)
    plt.ylabel("Fraction of labeled set")
    plt.title("Prediction Coverage and Unclear Rate")
    plt.legend()
    plt.tight_layout()
    plt.savefig(fig2, dpi=200)
    plt.close()

    # Confusion matrices
    cms = []
    counts = []
    for name, col in models.items():
        cm, cnt = confusion_matrix(df_eval["ground_truth"], df_eval[col])
        cms.append((name, cm))
        counts.append(cnt)

    plt.figure(figsize=(10, 8))
    for idx, (name, cm) in enumerate(cms, start=1):
        ax = plt.subplot(2, 2, idx)
        ax.imshow(cm)
        ax.set_title(f"{name}\\n(n={counts[idx-1]})")
        ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
        ax.set_xticklabels(["Pred Primary", "Pred Reuse"], rotation=30, ha="right")
        ax.set_yticklabels(["True Primary", "True Reuse"])
        for i in range(2):
            for j in range(2):
                ax.text(j, i, str(cm[i, j]), ha="center", va="center")
    plt.tight_layout()
    plt.savefig(fig3, dpi=200)
    plt.close()

    return {"accuracy": fig1, "coverage": fig2, "confusion": fig3}

# ---- Run evaluation if labeled CSV exists ----
if LABELED_CSV_PATH is None or not LABELED_CSV_PATH.exists():
    print("No labeled CSV provided. Skipping evaluation.")
else:
    # Manual CSV may not be utf-8
    gt = pd.read_csv(LABELED_CSV_PATH, encoding="latin1")

    if "ground_truth" not in gt.columns:
        if "human_label" in gt.columns:
            gt = gt.rename(columns={"human_label": "ground_truth"})
        else:
            raise ValueError("Labeled CSV must have 'human_label' or 'ground_truth'")

    # preserve legacy outputs if present (v1)
    if "heuristic_label" in gt.columns:
        gt = gt.rename(columns={"heuristic_label": "heuristic_v1_label"})
    if "llm_label" in gt.columns:
        gt = gt.rename(columns={"llm_label": "llm_v1_label"})

    # merge new outputs as v2
    df = gt.merge(
        preds.rename(columns={
            "heuristic_label": "heuristic_v2_label",
            "heuristic_conf":  "heuristic_v2_conf",
            "llm_label":       "llm_v2_label",
            "llm_conf":        "llm_v2_conf",
        })[["paper_id","heuristic_v2_label","heuristic_v2_conf","llm_v2_label","llm_v2_conf"]],
        on="paper_id",
        how="left"
    )

    for c in ["ground_truth","heuristic_v1_label","llm_v1_label","heuristic_v2_label","llm_v2_label"]:
        if c in df.columns:
            df[c] = df[c].apply(normalize_label)

    df_eval = df[df["ground_truth"].notna()].copy()

    models = {}
    if "heuristic_v1_label" in df_eval.columns:
        models["Heuristic v1 (title/early text)"] = "heuristic_v1_label"
    if "llm_v1_label" in df_eval.columns:
        models["LLM v1 (title/abstract)"] = "llm_v1_label"
    models["Heuristic v2 (evidence bundle)"] = "heuristic_v2_label"
    models["LLM v2 (evidence bundle)"] = "llm_v2_label"

    metrics = []
    for name, col in models.items():
        m = compute_metrics(df_eval["ground_truth"], df_eval[col])
        pred = df_eval[col]
        m["model"] = name
        m["coverage_binary"] = float(pred.isin(["Primary","Reuse"]).mean())
        m["unclear_rate"] = float((pred=="Unclear").mean())
        metrics.append(m)

    metrics_df = pd.DataFrame(metrics).set_index("model")
    metrics_csv = OUT_DIR / "model_comparison_metrics.csv"
    metrics_df.reset_index().to_csv(metrics_csv, index=False)

    merged_csv = OUT_DIR / "merged_eval.csv"
    df.to_csv(merged_csv, index=False)

    plot_paths = plot_outputs(metrics_df, df_eval, models, OUT_DIR)

    print("Saved metrics:", metrics_csv)
    print("Saved merged eval:", merged_csv)
    print("Saved plots:", plot_paths)
    display(metrics_df)


## Notes for improving accuracy

- Add **few-shot examples** from your manually labeled set into `build_prompt()`.
- Reduce abstentions by setting `LLM_MODE="all"` or by using LLM to decide only the uncertain cases.
- If you need binary-only output (no Unclear), adjust the prompt and/or post-process low-confidence predictions.
